# Principal components by power iteration

Here PCA is built without any eigenvalue routine: your code may not call `eig`, `eigh` or
`svd` from `np.linalg`, or anything from scikit-learn. Everything comes
from matrix-vector products. The data is a year of daily temperatures at eight weather
stations.

**Power iteration.** For a symmetric matrix $S$ with nonnegative eigenvalues and a single
largest one, start from a vector $\mathbf{v}$ and repeat

$$
\mathbf{v} \leftarrow \frac{S\mathbf{v}}{\lVert S\mathbf{v} \rVert} .
$$

Write $\mathbf{v}$ in the eigenvectors of $S$: each step multiplies the part along
eigenvector $i$ by $\lambda_i$, so after $t$ steps the ratio of any other part to the top
one has shrunk by $(\lambda_i / \lambda_1)^t$. The vector turns toward the top eigenvector,
and $\mathbf{v}^\top S\,\mathbf{v}$ toward the top eigenvalue.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says what
is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# 365 days of temperatures at 8 stations: a seasonal swing that all stations share, a
# north-south weather pattern, a coastal pattern, and local noise.
rng = np.random.default_rng(23)
days = np.arange(365)
season = 9 * np.sin(2 * np.pi * (days - 110) / 365)
weather = rng.normal(size=(365, 2)) * np.array([3.0, 1.5])
stations = np.array([[1.0, 0.9, 1.1, 0.8, 1.0, 1.2, 0.7, 0.9],
                     [1.0, 0.8, 0.4, 0.0, -0.3, -0.7, -0.9, -1.0],
                     [0.6, -0.5, 0.7, -0.6, 0.5, -0.4, 0.8, -0.7]])
X = np.c_[season, weather] @ stations + rng.normal(scale=0.8, size=(365, 8)) + np.linspace(8, 19, 8)
print("X", X.shape)

## Part 1 (coding task): The top eigenpair

Write `top_eigenpair(S, rng)` for a symmetric matrix `S` with nonnegative eigenvalues and a
single largest one. It returns `(lam, v)`: the largest eigenvalue and a unit eigenvector for
it, flipped if needed so that its entry of largest absolute value is positive. Run power
iteration until $\mathbf{v}$ changes by less than $10^{-12}$ in length from one step to the
next, or for at most 10,000 steps. `rng` is a NumPy random generator you may use.

The function must work for every such matrix, not only for covariance matrices of real
data.

In [ ]:
def top_eigenpair(S, rng):
    S = np.asarray(S, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
def _banned(*args, **kwargs):
    raise AssertionError("your code may not call an eigenvalue or SVD routine from np.linalg")

def _call(f, *args):
    saved = {name: getattr(np.linalg, name) for name in ("eig", "eigh", "eigvals", "eigvalsh", "svd")}
    for name in saved:
        setattr(np.linalg, name, _banned)
    try:
        return f(*args)
    finally:
        for name, routine in saved.items():
            setattr(np.linalg, name, routine)

def _signed(v):
    return v * np.sign(v[np.argmax(np.abs(v))])

def _reference(S):
    vals, vecs = np.linalg.eigh(S)
    order = np.argsort(vals)[::-1]
    return vals[order], np.array([_signed(vecs[:, i]) for i in order])

gen = np.random.default_rng(0)
lam, v = _call(top_eigenpair, np.diag([1.0, 5.0, 3.0]), gen)
assert np.isclose(lam, 5.0) and np.allclose(v, [0, 1, 0]), f"diag(1, 5, 3): got lam={lam}, v={v}, expected 5 and (0, 1, 0)"

S3 = np.array([[2.0, 0.0, -1.0], [0.0, 2.0, -1.0], [-1.0, -1.0, 3.0]])
lam, v = _call(top_eigenpair, S3, gen)
assert np.isclose(np.linalg.norm(v), 1.0), "v should have length 1"
assert np.isclose(lam, 4.0, atol=1e-8), \
    f"a 3 x 3 matrix with eigenvalues 4, 2 and 1: got lam={lam:.6f}. Make sure the iteration can reach the top eigenvector from your start"
assert np.allclose(v, np.array([-1.0, -1.0, 2.0]) / np.sqrt(6), atol=1e-6), f"v={np.round(v, 6)}: expected (-1, -1, 2) / sqrt(6)"

Xc = X - X.mean(axis=0)
C = Xc.T @ Xc / (len(X) - 1)
r_vals, r_vecs = _reference(C)
lam, v = _call(top_eigenpair, C, gen)
assert np.isclose(lam, r_vals[0], rtol=1e-9) and np.allclose(v, r_vecs[0], atol=1e-6), "wrong top eigenpair of the stations' covariance matrix"
print(f"stations: top eigenvalue {lam:.3f}")

close = np.diag([1.0, 0.95, 0.5])                    # the top two eigenvalues are close
lam, v = _call(top_eigenpair, close, gen)
assert np.allclose(v, [1, 0, 0], atol=1e-6), f"eigenvalues 1 and 0.95: v={np.round(v, 6)}, keep iterating until v itself stops moving"
print("Part 1 passed")

## Part 2 (coding task): The top k components

Write `top_components(X, k, rng)` for a data matrix `X` of shape `(n, d)`. It returns
`(mean, components, variances)`: the column means, a `(k, d)` array with the top $k$
principal components as rows, largest variance first, each signed as in Part 1, and the
$k$ variances along them, eigenvalues of the sample covariance matrix (dividing by
$n - 1$). Use `top_eigenpair` for every component.

In [ ]:
def top_components(X, k, rng):
    X = np.asarray(X, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
def _banned(*args, **kwargs):
    raise AssertionError("your code may not call an eigenvalue or SVD routine from np.linalg")

def _call(f, *args):
    saved = {name: getattr(np.linalg, name) for name in ("eig", "eigh", "eigvals", "eigvalsh", "svd")}
    for name in saved:
        setattr(np.linalg, name, _banned)
    try:
        return f(*args)
    finally:
        for name, routine in saved.items():
            setattr(np.linalg, name, routine)

def _signed(v):
    return v * np.sign(v[np.argmax(np.abs(v))])

def _reference(S):
    vals, vecs = np.linalg.eigh(S)
    order = np.argsort(vals)[::-1]
    return vals[order], np.array([_signed(vecs[:, i]) for i in order])

gen = np.random.default_rng(1)
mean, comp, var = _call(top_components, X, 3, gen)
Xc = X - X.mean(axis=0)
r_vals, r_vecs = _reference(Xc.T @ Xc / (len(X) - 1))
assert np.shape(comp) == (3, 8) and np.shape(var) == (3,), f"shapes {np.shape(comp)}, {np.shape(var)}: expected (3, 8) and (3,)"
assert np.allclose(mean, X.mean(axis=0)), "mean: expected the column means"
gram = comp @ comp.T
if np.isclose(abs(gram[0, 1]), 1.0, atol=1e-3):
    raise AssertionError("components 1 and 2 are the same vector: each new component must be orthogonal to the ones before it")
assert np.allclose(gram, np.eye(3), atol=1e-6), "the components should be orthonormal"
if np.allclose(var, r_vals[:3] * (len(X) - 1) / len(X), rtol=1e-6):
    raise AssertionError("variances: the sample covariance divides by n - 1, not n")
assert np.allclose(var, r_vals[:3], rtol=1e-8), f"variances {np.round(var, 4)}: expected {np.round(r_vals[:3], 4)}"
assert np.allclose(comp, r_vecs[:3], atol=1e-6), "components: expected the top three eigenvectors of the sample covariance, largest first"

S4 = np.array([[2.0, 0.0, -1.0], [0.0, 2.0, -1.0], [-1.0, -1.0, 3.0]])
rows = np.random.default_rng(5).multivariate_normal(np.zeros(3), S4, size=50)
_, c4, v4 = _call(top_components, rows, 3, gen)
r4_vals, r4_vecs = _reference(np.cov(rows.T))
assert np.allclose(v4, r4_vals, rtol=1e-6) and np.allclose(c4, r4_vecs, atol=1e-5), "wrong components on a 50-row sample with three dimensions"
print("stations: variances of the first three components", np.round(var, 3))
print("Part 2 passed")

## Part 3 (coding task): Stop at a fraction of the variance

Write `components_for_fraction(X, f, rng)` for $0 < f < 1$. It returns the smallest $k$
whose first $k$ components explain at least a fraction $f$ of the total variance. Find the
components one at a time with `top_eigenpair` and stop as soon as the fraction is reached:
on data with hundreds of columns and a few strong patterns, the later components are never
needed.

In [ ]:
def components_for_fraction(X, f, rng):
    X = np.asarray(X, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
def _banned(*args, **kwargs):
    raise AssertionError("your code may not call an eigenvalue or SVD routine from np.linalg")

def _call(f, *args):
    saved = {name: getattr(np.linalg, name) for name in ("eig", "eigh", "eigvals", "eigvalsh", "svd")}
    for name in saved:
        setattr(np.linalg, name, _banned)
    try:
        return f(*args)
    finally:
        for name, routine in saved.items():
            setattr(np.linalg, name, routine)

def _signed(v):
    return v * np.sign(v[np.argmax(np.abs(v))])

def _reference(S):
    vals, vecs = np.linalg.eigh(S)
    order = np.argsort(vals)[::-1]
    return vals[order], np.array([_signed(vecs[:, i]) for i in order])

Xc = X - X.mean(axis=0)
r_vals, _ = _reference(Xc.T @ Xc / (len(X) - 1))
cum = np.cumsum(r_vals) / r_vals.sum()

calls = 0
_power = top_eigenpair
def top_eigenpair(S, rng):
    global calls
    calls += 1
    return _power(S, rng)
try:
    for f in [0.5, cum[1] - 1e-6, cum[1] + 1e-6, 0.95]:
        expected = int(np.argmax(cum >= f)) + 1
        calls = 0
        got = _call(components_for_fraction, X, f, np.random.default_rng(2))
        if got == 1 and expected > 1:
            raise AssertionError(f"f={f:.4f}: returned 1. Divide the kept variance by the total variance of the data, not by the variance found so far")
        assert got == expected, f"f={f:.4f}: returned {got}, expected {expected}"
        assert calls <= expected, f"f={f:.4f}: found {calls} components to answer {expected}. Stop as soon as the fraction is reached"
finally:
    top_eigenpair = _power
print("cumulative fractions", np.round(cum, 3))
print("Part 3 passed")

## Solution

### Part 1: The top eigenpair

In [ ]:
def top_eigenpair(S, rng):
    S = np.asarray(S, dtype=float)
    v = rng.normal(size=len(S))                      # a random start has a part along every eigenvector
    v /= np.linalg.norm(v)
    for _ in range(10_000):
        w = S @ v
        w /= np.linalg.norm(w)
        done = np.linalg.norm(w - v) < 1e-12
        v = w
        if done:
            break
    v = v * np.sign(v[np.argmax(np.abs(v))])
    return float(v @ S @ v), v

**Why it works.** Power iteration can only grow the parts of the start vector that are already there. A
start with no part along the top eigenvector stays without one: the all-ones vector is an
eigenvector of the $3 \times 3$ check matrix, for its smallest eigenvalue 1, so from it every
step returns the same vector and the loop stops at once with $\lambda = 1$. A random start
has a nonzero part along every eigenvector with probability 1. The stopping rule watches
$\mathbf{v}$ and not $\lambda$: $\mathbf{v}^\top S\,\mathbf{v}$ settles about twice as fast,
so it can look converged while $\mathbf{v}$ is still turning, as it does for the close
eigenvalues 1 and 0.95, which take several hundred steps.

### Part 2: The top k components

In [ ]:
def top_components(X, k, rng):
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=0)
    Xc = X - mean
    S = Xc.T @ Xc / (len(X) - 1)
    components, variances = [], []
    for _ in range(k):
        lam, v = top_eigenpair(S, rng)
        components.append(v)
        variances.append(lam)
        S = S - lam * np.outer(v, v)                 # remove the direction just found
    return mean, np.array(components), np.array(variances)

**Why it works.** Once $(\lambda_1, \mathbf{v}_1)$ is known, $S - \lambda_1 \mathbf{v}_1\mathbf{v}_1^\top$
has the same eigenvectors as $S$ with $\lambda_1$ replaced by 0, because
$S = \sum_i \lambda_i \mathbf{v}_i \mathbf{v}_i^\top$. Its top eigenpair is
$(\lambda_2, \mathbf{v}_2)$, and repeating gives the components in order. Without that
subtraction every call finds $\mathbf{v}_1$ again. Each round inherits the rounding error
of the ones before, which is fine for a few components, and libraries use the SVD when they
need many.

### Part 3: Stop at a fraction of the variance

In [ ]:
def components_for_fraction(X, f, rng):
    X = np.asarray(X, dtype=float)
    Xc = X - X.mean(axis=0)
    S = Xc.T @ Xc / (len(X) - 1)
    total = np.trace(S)                              # the sum of all eigenvalues, without finding them
    kept = 0.0
    for k in range(1, len(S) + 1):
        lam, v = top_eigenpair(S, rng)
        kept += lam
        if kept >= f * total:
            return k
        S = S - lam * np.outer(v, v)
    return len(S)

**Why it works.** The fraction needs the total variance, and the total is the sum of all $d$ eigenvalues,
which is the trace of the covariance matrix: the sum of the column variances, available
before any eigenvector is found. So the loop can stop at the first $k$ that reaches the
target. On the stations the seasonal swing, which every station shares, explains about
0.85 of the variance, the north-south pattern takes it to 0.97 and the coastal pattern to
0.99. A fraction of 0.95 needs two components, and the other six are never computed.